# AQI Feature Store — Exploratory Data Analysis

Covers all 10 cities in `config.CITIES` over the full backfilled history (currently ~365 days). Run after `python -m feature_pipeline.backfill_pipeline` has populated the Hopsworks feature group, or via `.github/workflows/run_eda.yml` (executes headlessly, uploads this notebook with real outputs as a workflow artifact).

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from hopsworks_utils import get_feature_group

fg = get_feature_group()
df = fg.read()
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)
df = df.sort_values(['city', 'event_time']).reset_index(drop=True)

city_order = list(config.CITIES.keys())
city_labels = {k: v['label'] for k, v in config.CITIES.items()}

print(f'{len(df)} rows across {df["city"].nunique()} cities, '
      f'{df["event_time"].min()} to {df["event_time"].max()}')

## Missing values

By column overall, then by city — a city with a much higher null rate than the rest usually means a data-quality problem specific to it (this is exactly how the Sukkur → Rawalpindi swap got flagged).

In [ ]:
df.isna().mean().sort_values(ascending=False)

In [ ]:
missing_by_city = df.groupby('city').apply(lambda g: g['us_aqi'].isna().mean())
missing_by_city = missing_by_city.reindex(city_order).rename(index=city_labels)
missing_by_city.sort_values(ascending=False)

## Which cities have the worst air quality on average?

Ranks cities by mean/median/max US AQI over the full backfilled window — the single most direct answer to "which city is more polluted."

In [ ]:
city_stats = (
    df.groupby('city')['us_aqi']
    .agg(['mean', 'median', 'std', 'max'])
    .reindex(city_order)
    .rename(index=city_labels)
    .sort_values('mean', ascending=False)
)
city_stats

## AQI distribution by city

Boxplot, ordered by median — shows both typical level and spread/volatility per city, not just the mean.

In [ ]:
order = city_stats.sort_values('median', ascending=False).index.tolist()
data_by_city = [df[df['city'].map(city_labels) == c]['us_aqi'].dropna() for c in order]

fig, ax = plt.subplots(figsize=(12, 5))
ax.boxplot(data_by_city, tick_labels=order, showfliers=False)
ax.set_title('US AQI distribution by city (outliers hidden for readability)')
ax.set_ylabel('US AQI')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## AQI trend over time, per city

One panel per city so a spike in one city doesn't compress the y-axis for the rest.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 16), sharex=True)
for ax, city_key in zip(axes.flat, city_order):
    city_df = df[df['city'] == city_key]
    ax.plot(city_df['event_time'], city_df['us_aqi'], linewidth=0.7)
    ax.set_title(city_labels[city_key])
    ax.set_ylabel('US AQI')
plt.tight_layout()
plt.show()

## Seasonality: average AQI by hour of day (per city)

Heatmap — rows are cities, columns are hour-of-day (UTC), color is mean US AQI.

In [ ]:
def seasonality_heatmap(group_col, title, xlabel):
    pivot = (
        df.groupby(['city', group_col])['us_aqi']
        .mean()
        .unstack(group_col)
        .reindex(city_order)
        .rename(index=city_labels)
    )
    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label='Mean US AQI')
    plt.tight_layout()
    plt.show()
    return pivot

_ = seasonality_heatmap('hour', 'Average AQI by hour of day (UTC), per city', 'Hour')

## Seasonality: average AQI by month

Only meaningful now that the backfill covers a full year (a 90-day window couldn't show a full seasonal cycle).

In [ ]:
_ = seasonality_heatmap('month', 'Average AQI by month, per city', 'Month')

## Seasonality: average AQI by day of week

0 = Monday.

In [ ]:
_ = seasonality_heatmap('day_of_week', 'Average AQI by day of week, per city', 'Day of week (0=Mon)')

## Correlation between pollutants/weather and AQI

Pooled across all cities first, then per-city — a feature that correlates well overall but not in a specific city is a clue that city needs different features/handling.

In [ ]:
numeric_cols = df.select_dtypes('number').columns
corr = df[numeric_cols].corr()['us_aqi'].sort_values(ascending=False)
corr

In [ ]:
feature_cols = [c for c in numeric_cols if c != 'us_aqi']
per_city_corr = (
    df.groupby('city')[list(numeric_cols)]
    .apply(lambda g: g.corr()['us_aqi'].drop('us_aqi'))
    .reindex(city_order)
    .rename(index=city_labels)
)
per_city_corr

## Key findings

_Filled in after running this notebook against real Hopsworks data (via `.github/workflows/run_eda.yml` or locally) — see the executed version's outputs above for the actual numbers this summarizes._

- **Missing data:** ...
- **Most/least polluted cities:** ...
- **Daily pattern:** ...
- **Seasonal pattern:** ...
- **Strongest predictors of AQI:** ...
